# Notebook 16: Dice Loss

**Module:** 07 Segmentation  
**Duration:** ~2 hrs

---

## Learning Objectives

1. Derive Dice coefficient and loss
2. Implement dice_with_logits (your losses.py)
3. Understand why Dice handles imbalance

## Dice Coefficient

$$\text{Dice} = \frac{2|A \cap B|}{|A| + |B|}$$

$$L_{\text{Dice}} = 1 - \text{Dice}$$

**Soft Dice (differentiable):** Use sigmoid probabilities instead of binary.

**Your losses.py:**
```python
def dice_with_logits(logits, targets):
    probs = torch.sigmoid(logits)
    inter = (probs * targets).sum(dim=(2,3))
    union = probs.sum(dim=(2,3)) + targets.sum(dim=(2,3))
    return (2*inter + 1) / (union + 1)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
def pixel_accuracy(pred, target):
    return (pred == target).float().mean().item()

def iou_score(pred, target, n_classes=None, smooth=1e-7):
    """pred, target: (N,H,W) long tensors"""
    if n_classes is None:
        n_classes = int(max(pred.max(), target.max())) + 1
    ious = []
    for c in range(n_classes):
        p = (pred == c)
        t = (target == c)
        inter = (p & t).sum().float()
        union = (p | t).sum().float()
        if union > 0:
            ious.append(((inter + smooth) / (union + smooth)).item())
    return float(np.mean(ious)) if ious else 0.0

def dice_score(pred, target, smooth=1e-7):
    """Binary pred, target: (N,1,H,W) float"""
    inter = (pred * target).sum(dim=(2,3))
    return ((2*inter + smooth) / (pred.sum(dim=(2,3)) + target.sum(dim=(2,3)) + smooth)).mean().item()

In [ ]:
# Reimplement your water-bodies dice_with_logits
def dice_with_logits(logits, targets, smooth=1.0):
    probs = torch.sigmoid(logits)
    t = targets.float()
    inter = (probs * t).sum(dim=(2, 3))
    union = probs.sum(dim=(2, 3)) + t.sum(dim=(2, 3))
    return ((2.0 * inter + smooth) / (union + smooth)).mean()

logits = torch.randn(4, 1, 32, 32)
targets = (torch.rand(4, 1, 32, 32) > 0.7).float()
print(f'Dice score: {dice_with_logits(logits, targets):.4f}')
print(f'Dice loss: {1 - dice_with_logits(logits, targets):.4f}')

---

## Summary

Dice loss handles class imbalance — essential for small water bodies in large tiles.

**Next:** [17_IoU_Loss.ipynb](17_IoU_Loss.ipynb)